[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/03_gradient_descent_and_convergence/exercises.ipynb)

# Exercises — Gradient Descent and Convergence

Twenty fully solved problems in four tiers. Every numeric answer is recomputed by the code cell
that follows its solution; none is quoted from memory.

In [1]:
import numpy as np
from scipy import linalg as sla
from scipy import optimize as sopt

rng = np.random.default_rng(0)
np.set_printoptions(precision=6, suppress=True)


def gd_path(grad, x0, alpha, n_steps):
    """Gradient-descent trajectory as an (n_steps + 1, n) array."""
    xs = [np.asarray(x0, dtype=float)]
    for _ in range(n_steps):
        xs.append(xs[-1] - alpha * grad(xs[-1]))
    return np.array(xs)


def iters_per_digit(rate):
    """Iterations needed to shrink a quantity contracting by `rate` per step tenfold."""
    return np.log(10) / -np.log(rate)


print("helpers ready")

helpers ready


## L0 — Concept Checks

### Problem L0.1 — Why the Negative Gradient

**Statement.** Prove that among all unit vectors $\mathbf{d} \in \mathbb{R}^n$, the directional
derivative $\nabla f(\mathbf{x})^T\mathbf{d}$ is minimized by
$\mathbf{d} = -\nabla f(\mathbf{x})/\lVert \nabla f(\mathbf{x})\rVert$, assuming
$\nabla f(\mathbf{x}) \neq \mathbf{0}$.

**Intuition.** The gradient is the direction of steepest ascent, so the steepest descent direction
should be its exact opposite; Cauchy-Schwarz turns "should" into "is".

**Solution.**

**Step 1.** By Cauchy-Schwarz, for any unit $\mathbf{d}$,

$$
\left\lvert \nabla f(\mathbf{x})^T\mathbf{d}\right\rvert \le \lVert \nabla f(\mathbf{x})\rVert\,\lVert \mathbf{d}\rVert = \lVert \nabla f(\mathbf{x})\rVert
$$

so $\nabla f(\mathbf{x})^T\mathbf{d} \ge -\lVert \nabla f(\mathbf{x})\rVert$.

**Step 2.** Equality in Cauchy-Schwarz holds if and only if $\mathbf{d}$ is a scalar multiple of
$\nabla f(\mathbf{x})$. Achieving the *lower* value forces a negative multiple, and
$\lVert \mathbf{d}\rVert = 1$ fixes it uniquely.

**Step 3.** With that choice the value is $-\lVert \nabla f(\mathbf{x})\rVert$, exactly the bound of
Step 1, so the minimum is attained.

$$
\boxed{\mathbf{d}^{*} = -\frac{\nabla f(\mathbf{x})}{\lVert \nabla f(\mathbf{x})\rVert}, \qquad \nabla f(\mathbf{x})^T\mathbf{d}^{*} = -\lVert \nabla f(\mathbf{x})\rVert}
$$

**Key takeaway.** Steepest descent is one line of Cauchy-Schwarz — and it is norm-dependent, which
is the seed of preconditioning and of Newton's method.

In [2]:
g_probe = np.array([3.0, -1.0, 2.0])
d_star = -g_probe / np.linalg.norm(g_probe)
d_rand = rng.standard_normal((20000, 3))
d_rand /= np.linalg.norm(d_rand, axis=1, keepdims=True)
vals = d_rand @ g_probe
print(f"grad = {g_probe},  |grad| = {np.linalg.norm(g_probe):.6f}")
print(f"value at d*                = {g_probe @ d_star:.6f}")
print(f"best over 20000 random unit d = {vals.min():.6f}")
assert g_probe @ d_star <= vals.min() + 1e-12

grad = [ 3. -1.  2.],  |grad| = 3.741657
value at d*                = -3.741657
best over 20000 random unit d = -3.741644


### Problem L0.2 — Does Gradient Descent Always Descend

**Statement.** True or false: for an $L$-smooth function, gradient descent with any fixed step
$\alpha \gt 0$ decreases $f$ at every iteration. Give a proof or a counterexample, and state the
sharp threshold on $\alpha$.

**Intuition.** The linear model that justifies stepping downhill is trustworthy only over a
distance set by the curvature bound $L$.

**Solution.**

**Step 1 (counterexample).** Take $f(x) = \frac{L}{2}x^2$, which is $L$-smooth with $f'(x) = Lx$.
Gradient descent gives $x_{k+1} = (1-\alpha L)x_k$. With $\alpha = 3/L$ this is $x_{k+1} = -2x_k$,
so $f(x_k) = 4^k f(x_0) \to \infty$: the objective *increases* at every step. The claim is **false**.

**Step 2 (where the threshold comes from).** The descent lemma gives

$$
f(x_{k+1}) \le f(x_k) - \alpha\left(1 - \frac{\alpha L}{2}\right)\lVert \nabla f(x_k)\rVert^2
$$

and the coefficient $\alpha(1-\alpha L/2)$ is positive exactly when $0 \lt \alpha \lt 2/L$.

**Step 3 (the threshold is sharp).** On the same quadratic, $\lvert 1-\alpha L\rvert \ge 1$ once
$\alpha \ge 2/L$, so the iterates fail to converge; at $\alpha = 2/L$ they oscillate forever with
constant amplitude. Hence no larger range works for all $L$-smooth $f$.

$$
\boxed{0 \lt \alpha \lt \frac{2}{L} \quad\text{is necessary and sufficient for guaranteed descent}}
$$

**Key takeaway.** "Downhill direction" does not imply "downhill step": descent is a joint property
of the direction and of a step size compatible with $L$.

In [3]:
L0 = 4.0
f_q1 = lambda x: 0.5 * L0 * x ** 2
for alpha in [0.5 / L0, 1.0 / L0, 1.9 / L0, 2.0 / L0, 3.0 / L0]:
    xs = np.array([1.0])
    x = 1.0
    for _ in range(10):
        x = x - alpha * L0 * x
        xs = np.append(xs, x)
    print(f"alpha*L = {alpha * L0:.2f}   f(x_10)/f(x_0) = {f_q1(xs[-1]) / f_q1(xs[0]):.6e}"
          f"   monotone decrease: {np.all(np.diff(f_q1(xs)) <= 1e-15)}")
assert f_q1(1.0 * (1 - 3.0) ** 10) > f_q1(1.0)

alpha*L = 0.50   f(x_10)/f(x_0) = 9.536743e-07   monotone decrease: True
alpha*L = 1.00   f(x_10)/f(x_0) = 0.000000e+00   monotone decrease: True
alpha*L = 1.90   f(x_10)/f(x_0) = 1.215767e-01   monotone decrease: True
alpha*L = 2.00   f(x_10)/f(x_0) = 1.000000e+00   monotone decrease: True
alpha*L = 3.00   f(x_10)/f(x_0) = 1.048576e+06   monotone decrease: False


### Problem L0.3 — Reading the Condition Number

**Statement.** For $f(\mathbf{x}) = \frac12\mathbf{x}^TA\mathbf{x}$ with
$0 \prec \mu I \preceq A \preceq LI$, define $\kappa$ and say what $\kappa = 1$ and $\kappa \gg 1$
mean for the level sets and for gradient descent.

**Intuition.** Level sets of a quadratic are ellipsoids whose axis lengths come from the
eigenvalues of $A$.

**Solution.**

**Step 1 (definition).** $\kappa = L/\mu = \lambda_{\max}(A)/\lambda_{\min}(A) \ge 1$.

**Step 2 (geometry).** The level set $\{f = c\}$ is an ellipsoid whose semi-axis along the
eigenvector of $\lambda_i$ has length $\sqrt{2c/\lambda_i}$, so the aspect ratio is $\sqrt{\kappa}$.

**Step 3 ($\kappa = 1$).** Then $A = \mu I$, level sets are spheres, the gradient
$A\mathbf{x} = \mu\mathbf{x}$ points exactly at the minimizer, and one step with $\alpha = 1/\mu$
lands on it.

**Step 4 ($\kappa \gg 1$).** Long thin ellipsoids: the gradient is nearly orthogonal to the long
axis, the iterates zig-zag across the valley and creep along it, and the optimal contraction
$\frac{\kappa-1}{\kappa+1} \to 1$.

$$
\boxed{\kappa = \frac{\lambda_{\max}(A)}{\lambda_{\min}(A)}; \quad \kappa = 1 \Rightarrow \text{one-step convergence}, \quad \kappa \gg 1 \Rightarrow \text{zig-zag at rate } \frac{\kappa-1}{\kappa+1}}
$$

**Key takeaway.** Gradient descent does not see the minimizer, it sees the normal to the level set;
only spherical geometry makes the two coincide.

In [4]:
for kap in [1.0, 4.0, 100.0]:
    Ak = np.diag([1.0, kap])
    x0k = np.array([1.0, 1.0])
    alpha_k = 2 / (1.0 + kap)
    xs = gd_path(lambda x, A=Ak: A @ x, x0k, alpha_k, 30)
    ang = np.degrees(np.arccos(np.clip((Ak @ x0k) @ (-x0k)
                                       / (np.linalg.norm(Ak @ x0k) * np.linalg.norm(x0k)), -1, 1)))
    print(f"kappa={kap:6.1f}  aspect ratio sqrt(kappa)={np.sqrt(kap):6.3f}  "
          f"angle(-grad, direction to minimizer)={ang:6.2f} deg  "
          f"|x_30|/|x_0|={np.linalg.norm(xs[-1]) / np.linalg.norm(xs[0]):.3e}  "
          f"predicted {(kap - 1) / (kap + 1):.4f}^30={((kap - 1) / (kap + 1)) ** 30:.3e}")

kappa=   1.0  aspect ratio sqrt(kappa)= 1.000  angle(-grad, direction to minimizer)=180.00 deg  |x_30|/|x_0|=0.000e+00  predicted 0.0000^30=0.000e+00
kappa=   4.0  aspect ratio sqrt(kappa)= 2.000  angle(-grad, direction to minimizer)=149.04 deg  |x_30|/|x_0|=2.211e-07  predicted 0.6000^30=2.211e-07
kappa= 100.0  aspect ratio sqrt(kappa)=10.000  angle(-grad, direction to minimizer)=135.57 deg  |x_30|/|x_0|=5.488e-01  predicted 0.9802^30=5.488e-01


### Problem L0.4 — PL and Stationary Points

**Statement.** Suppose $f$ satisfies PL,
$\frac12\lVert \nabla f(\mathbf{x})\rVert^2 \ge \mu(f(\mathbf{x}) - f^{*})$ with $\mu \gt 0$. Prove
every stationary point is a global minimizer. Is every PL function convex?

**Intuition.** PL says gradients cannot vanish while the value is still above the minimum.

**Solution.**

**Step 1.** Let $\nabla f(\bar{\mathbf{x}}) = \mathbf{0}$. Substituting into PL,

$$
0 = \tfrac12\lVert \nabla f(\bar{\mathbf{x}})\rVert^2 \ge \mu\left(f(\bar{\mathbf{x}}) - f^{*}\right) \ge 0
$$

**Step 2.** Since $\mu \gt 0$, both inequalities are equalities and $f(\bar{\mathbf{x}}) = f^{*}$.

**Step 3 (PL does not imply convexity).** $f(x) = x^2 + 3\sin^2 x$ has
$f''(x) = 2 + 6\cos 2x \lt 0$ whenever $\cos 2x \lt -1/3$, so $f$ is non-convex; yet $f^{*} = 0$ and
$\frac12(f'(x))^2 \ge \frac{1}{32}f(x)$ for all $x$, so $f$ is PL. (The code below measures the
sharp constant.)

$$
\boxed{\text{PL} \Rightarrow \text{every stationary point is a global minimum, but PL} \not\Rightarrow \text{convexity}}
$$

**Key takeaway.** PL is a landscape condition, not a curvature condition: the terrain may ripple,
but no ripple is deep enough to create a spurious valley.

In [5]:
f_pl = lambda x: x ** 2 + 3 * np.sin(x) ** 2
fp_pl = lambda x: 2 * x + 3 * np.sin(2 * x)
fpp_pl = lambda x: 2 + 6 * np.cos(2 * x)

xs = np.linspace(-30, 30, 600001)
xs = xs[np.abs(f_pl(xs)) > 1e-12]
ratio = 0.5 * fp_pl(xs) ** 2 / f_pl(xs)
print(f"min f''  over the grid = {fpp_pl(xs).min():.4f}  -> non-convex")
print(f"sharp PL constant mu   = {ratio.min():.6f}  at x = {xs[ratio.argmin()]:.4f}")
print(f"claimed constant 1/32  = {1 / 32:.6f}  -> claim is valid and conservative")
assert ratio.min() >= 1 / 32
grid = np.linspace(-10, 10, 200001)
flat = grid[np.abs(fp_pl(grid)) < 1e-3]
print(f"points with |f'| < 1e-3 on [-10,10]: {flat}  -> the only stationary point is x = 0, "
      f"where f = {f_pl(0.0):.1f} = f*")
assert np.all(np.abs(flat) < 1e-3)

min f''  over the grid = -4.0000  -> non-convex
sharp PL constant mu   = 0.175531  at x = 2.2017
claimed constant 1/32  = 0.031250  -> claim is valid and conservative
points with |f'| < 1e-3 on [-10,10]: [-0.0001  0.      0.0001]  -> the only stationary point is x = 0, where f = 0.0 = f*


## L1 — Foundations

### Problem L1.1 — Rates from a Spectrum

**Statement.** Let $f(\mathbf{x}) = \frac12\mathbf{x}^TA\mathbf{x}$ with
$A = \begin{bmatrix}3 & 1\\ 1 & 3\end{bmatrix}$. Find $L$, $\mu$, $\kappa$, the optimal fixed step
and the resulting per-iteration contraction factor of the error.

**Intuition.** For quadratics everything is spectral: two eigenvalues determine the whole
convergence story.

**Solution.**

**Step 1 (eigenvalues).** $\det(A-\lambda I) = (3-\lambda)^2 - 1 = 0$ gives $3-\lambda = \pm1$, so
$\lambda = 2$ with eigenvector $(1,-1)^T$ and $\lambda = 4$ with eigenvector $(1,1)^T$.

**Step 2 (constants).** $\mu = 2$, $L = 4$, $\kappa = L/\mu = 2$.

**Step 3 (optimal step and rate).**

$$
\alpha^{*} = \frac{2}{L+\mu} = \frac{2}{6} = \frac13, \qquad \rho = \frac{\kappa-1}{\kappa+1} = \frac13
$$

**Step 4 (check).** At $\alpha = 1/3$ the mode factors are $1 - \frac23 = \frac13$ and
$1 - \frac43 = -\frac13$: equal in modulus, so the two extreme modes are balanced, which is exactly
the optimality condition.

$$
\boxed{\mu = 2,\quad L = 4,\quad \kappa = 2,\quad \alpha^{*} = \tfrac13,\quad \rho = \tfrac13}
$$

**Key takeaway.** Diagonalize once and gradient descent on a quadratic becomes $n$ independent
scalar recursions whose slowest and most oscillatory modes are the extreme eigenvalues.

In [6]:
A11 = np.array([[3.0, 1.0], [1.0, 3.0]])
ev, Q = np.linalg.eigh(A11)
mu11, L11 = ev.min(), ev.max()
kap11 = L11 / mu11
alpha11 = 2 / (L11 + mu11)
xs = gd_path(lambda x: A11 @ x, np.array([1.0, 0.0]), alpha11, 8)
ratios = np.linalg.norm(xs[1:], axis=1) / np.linalg.norm(xs[:-1], axis=1)
print(f"eigenvalues {ev}   eigenvectors\n{Q}")
print(f"mu={mu11}  L={L11}  kappa={kap11}  alpha*={alpha11:.6f}  rho={(kap11 - 1) / (kap11 + 1):.6f}")
print(f"mode factors 1-alpha*lambda = {1 - alpha11 * ev}")
print(f"measured error ratios       = {ratios}")
print(f"iterations per digit (error) = {iters_per_digit((kap11 - 1) / (kap11 + 1)):.4f}")
assert np.allclose(ratios, (kap11 - 1) / (kap11 + 1))

eigenvalues [2. 4.]   eigenvectors
[[-0.707107  0.707107]
 [ 0.707107  0.707107]]
mu=2.0  L=4.0  kappa=2.0  alpha*=0.333333  rho=0.333333
mode factors 1-alpha*lambda = [ 0.333333 -0.333333]
measured error ratios       = [0.333333 0.333333 0.333333 0.333333 0.333333 0.333333 0.333333 0.333333]
iterations per digit (error) = 2.0959


### Problem L1.2 — One Step by Hand on an Ill-Conditioned Bowl

**Statement.** For $f(x,y) = x^2 + 10y^2$, take one gradient step from $(1,1)$ with $\alpha = 0.1$.
Compare $f$ before and after, describe each coordinate, and find the step range for which both
coordinates converge.

**Intuition.** The $y$-direction has ten times the curvature of $x$, so one shared step size must
serve two very different masters.

**Solution.**

**Step 1 (derivatives).** $\nabla f(x,y) = (2x, 20y)$, so $\nabla f(1,1) = (2,20)$; the Hessian is
$\operatorname{diag}(2,20)$, giving $\mu = 2$, $L = 20$, $\kappa = 10$.

**Step 2 (the step).** $(x_1,y_1) = (1,1) - 0.1(2,20) = (0.8,\,-1.0)$.

**Step 3 (values).** $f(1,1) = 11$ and $f(0.8,-1) = 0.64 + 10 = 10.64$: a decrease of $0.36$ only.

**Step 4 (mode analysis).** $x_{k+1} = (1-2\alpha)x_k = 0.8x_k$ and
$y_{k+1} = (1-20\alpha)y_k = -1.0\,y_k$: with $\alpha = 0.1$ the $y$-mode has factor exactly $-1$,
so it flips sign forever with undiminished amplitude and never converges.

**Step 5 (safe range).** Both $\lvert 1-2\alpha\rvert \lt 1$ and $\lvert 1-20\alpha\rvert \lt 1$
require $0 \lt \alpha \lt 2/L = 0.1$, with optimum $\alpha^{*} = 2/(L+\mu) = 1/11$.

$$
\boxed{0 \lt \alpha \lt \frac{2}{L} = 0.1, \qquad \alpha^{*} = \frac{2}{L+\mu} = \frac{1}{11} \approx 0.090909}
$$

**Key takeaway.** The stiffest eigenvalue alone sets the stability ceiling; at $\alpha = 2/L$ the
stiff mode neither grows nor decays, so the loss stalls above its minimum while the soft mode
quietly converges.

In [7]:
f12 = lambda p: p[0] ** 2 + 10 * p[1] ** 2
g12 = lambda p: np.array([2 * p[0], 20 * p[1]])
mu12, L12 = 2.0, 20.0
p0 = np.array([1.0, 1.0])
p1 = p0 - 0.1 * g12(p0)
print(f"grad at (1,1) = {g12(p0)}   x1 = {p1}")
print(f"f before = {f12(p0):.4f}   f after = {f12(p1):.4f}   drop = {f12(p0) - f12(p1):.4f}")
print(f"mode factors at alpha=0.1: {1 - 0.1 * np.array([mu12, L12])}")
long_run = gd_path(g12, p0, 0.1, 200)
print(f"after 200 steps at alpha=0.1: x = {long_run[-1]},  f = {f12(long_run[-1]):.6f} (stalled)")
print(f"2/L = {2 / L12:.6f},  alpha* = {2 / (L12 + mu12):.6f} = 1/11")
opt_run = gd_path(g12, p0, 2 / (L12 + mu12), 200)
print(f"after 200 steps at alpha*:   f = {f12(opt_run[-1]):.3e}")
assert abs(f12(long_run[-1]) - 10.0) < 1e-9

grad at (1,1) = [ 2. 20.]   x1 = [ 0.8 -1. ]
f before = 11.0000   f after = 10.6400   drop = 0.3600
mode factors at alpha=0.1: [ 0.8 -1. ]
after 200 steps at alpha=0.1: x = [0. 1.],  f = 10.000000 (stalled)
2/L = 0.100000,  alpha* = 0.090909 = 1/11
after 200 steps at alpha*:   f = 1.518e-34


### Problem L1.3 — Descent Directions Decrease $f$

**Statement.** Let $f \in \mathcal{C}^1$ and suppose $\nabla f(\mathbf{x})^T\mathbf{d} \lt 0$.
Prove there is $\bar{\alpha} \gt 0$ with $f(\mathbf{x}+\alpha\mathbf{d}) \lt f(\mathbf{x})$ for all
$\alpha \in (0,\bar{\alpha})$.

**Intuition.** A negative slope at $\alpha = 0$ must push the one-dimensional restriction below its
starting value, at least briefly.

**Solution.**

**Step 1 (reduce to one dimension).** Put $\phi(\alpha) = f(\mathbf{x}+\alpha\mathbf{d})$. By the
chain rule $\phi'(0) = \nabla f(\mathbf{x})^T\mathbf{d} \lt 0$.

**Step 2 (definition of the derivative).** Since
$\lim_{\alpha\to0^{+}}\frac{\phi(\alpha)-\phi(0)}{\alpha} = \phi'(0)$, taking
$\epsilon = -\phi'(0)/2 \gt 0$ produces $\bar{\alpha} \gt 0$ such that for all
$\alpha \in (0,\bar{\alpha})$,

$$
\frac{\phi(\alpha)-\phi(0)}{\alpha} \lt \phi'(0) + \epsilon = \frac{\phi'(0)}{2} \lt 0
$$

**Step 3 (conclude).** Multiply by $\alpha \gt 0$.

$$
\boxed{f(\mathbf{x}+\alpha\mathbf{d}) \lt f(\mathbf{x}) + \frac{\alpha}{2}\nabla f(\mathbf{x})^T\mathbf{d} \lt f(\mathbf{x}) \qquad \text{for all } \alpha \in (0,\bar{\alpha})}
$$

**Key takeaway.** Every direction in the open half-space $\nabla f^T\mathbf{d} \lt 0$ works locally;
steepest descent is the *best* such direction, not the only one, and that freedom is what line
searches exploit.

In [8]:
f13 = lambda p: p[0] ** 2 + 10 * p[1] ** 2 + 0.5 * p[0] * p[1]
g13 = lambda p: np.array([2 * p[0] + 0.5 * p[1], 20 * p[1] + 0.5 * p[0]])
x13 = np.array([1.0, 0.5])
grad = g13(x13)
for name, d in [("steepest", -grad),
                ("random descent", None),
                ("ascent", grad)]:
    if d is None:
        while True:
            d = rng.standard_normal(2)
            if grad @ d < 0:
                break
    d = d / np.linalg.norm(d)              # unit directions, so alpha is a true distance
    slope = grad @ d
    alphas = np.array([1e-1, 1e-2, 1e-3, 1e-4])
    drops = np.array([f13(x13 + a * d) - f13(x13) for a in alphas])
    print(f"{name:<16} slope = {slope:+.4f}   f(x+ad)-f(x) at a={alphas} -> {drops}")
    if slope < 0:
        assert np.all(drops[1:] < 0)

steepest         slope = -10.7384   f(x+ad)-f(x) at a=[0.1    0.01   0.001  0.0001] -> [-0.976763 -0.106413 -0.010729 -0.001074]
random descent   slope = -2.9504   f(x+ad)-f(x) at a=[0.1    0.01   0.001  0.0001] -> [-0.284295 -0.029396 -0.002949 -0.000295]
ascent           slope = +10.7384   f(x+ad)-f(x) at a=[0.1    0.01   0.001  0.0001] -> [1.17091  0.108354 0.010748 0.001074]


### Problem L1.4 — Exact Solution of Gradient Descent in One Dimension

**Statement.** For $f(x) = \frac{L}{2}x^2$ solve the gradient-descent recursion in closed form for
fixed $\alpha \gt 0$, and classify convergence, oscillation and divergence in terms of $\alpha L$.

**Intuition.** One dimension exposes the full phase diagram of step-size behaviour with no linear
algebra in the way.

**Solution.**

**Step 1 (closed form).** $f'(x) = Lx$, so $x_{k+1} = (1-\alpha L)x_k$ and

$$
x_k = (1-\alpha L)^k x_0, \qquad f(x_k) = (1-\alpha L)^{2k} f(x_0)
$$

**Step 2 (phase diagram).** With $r = 1-\alpha L$:

| regime | $r$ | behaviour |
|---|---|---|
| $0 \lt \alpha L \lt 1$ | $r \in (0,1)$ | monotone geometric convergence, no sign change |
| $\alpha L = 1$ | $r = 0$ | exact convergence in one step |
| $1 \lt \alpha L \lt 2$ | $r \in (-1,0)$ | convergent but oscillating across the minimum |
| $\alpha L = 2$ | $r = -1$ | bounded oscillation forever, no convergence |
| $\alpha L \gt 2$ | $\lvert r\rvert \gt 1$ | geometric divergence with sign flips |

**Step 3 (values versus iterates).** $f(x_k)$ is monotone in every regime, decaying at rate $r^2$
or exploding at rate $r^2 \gt 1$ — a reminder that the value rate is the *square* of the iterate
rate.

$$
\boxed{x_k = (1-\alpha L)^k x_0:\ \text{converges iff } 0 \lt \alpha L \lt 2,\ \text{oscillates for } \alpha L \gt 1,\ \text{diverges for } \alpha L \gt 2}
$$

**Key takeaway.** The scalar factor $1-\alpha\lambda$ is the atom of all gradient-descent analysis:
every multidimensional quadratic is $n$ copies of this problem running in parallel.

In [9]:
L14 = 5.0
for aL in [0.5, 1.0, 1.5, 2.0, 2.5]:
    alpha = aL / L14
    x = 1.0
    seq = [x]
    for _ in range(8):
        x = x - alpha * L14 * x
        seq.append(x)
    seq = np.array(seq)
    closed = (1 - aL) ** np.arange(9)
    print(f"aL={aL:.1f}  r={1 - aL:+.1f}  x_k={seq}  matches closed form: {np.allclose(seq, closed)}")
    assert np.allclose(seq, closed)

aL=0.5  r=+0.5  x_k=[1.       0.5      0.25     0.125    0.0625   0.03125  0.015625 0.007812
 0.003906]  matches closed form: True
aL=1.0  r=+0.0  x_k=[1. 0. 0. 0. 0. 0. 0. 0. 0.]  matches closed form: True
aL=1.5  r=-0.5  x_k=[ 1.       -0.5       0.25     -0.125     0.0625   -0.03125   0.015625
 -0.007812  0.003906]  matches closed form: True
aL=2.0  r=-1.0  x_k=[ 1. -1.  1. -1.  1. -1.  1. -1.  1.]  matches closed form: True
aL=2.5  r=-1.5  x_k=[  1.        -1.5        2.25      -3.375      5.0625    -7.59375
  11.390625 -17.085938  25.628906]  matches closed form: True


### Problem L1.5 — Strong Convexity Implies the PL Inequality

**Statement.** Prove that a differentiable $\mu$-strongly convex $f$ with minimizer
$\mathbf{x}^{*}$ satisfies $\frac12\lVert \nabla f(\mathbf{x})\rVert^2 \ge \mu(f(\mathbf{x})-f^{*})$.

**Intuition.** Strong convexity gives a quadratic lower bound; minimizing both sides of that bound
converts it into a gradient-dominance statement.

**Solution.**

**Step 1.** Strong convexity says, for all $\mathbf{x},\mathbf{y}$,

$$
f(\mathbf{y}) \ge f(\mathbf{x}) + \nabla f(\mathbf{x})^T(\mathbf{y}-\mathbf{x}) + \frac{\mu}{2}\lVert \mathbf{y}-\mathbf{x}\rVert^2
$$

**Step 2 (minimize the right side over $\mathbf{y}$).** It is a strictly convex quadratic whose
gradient vanishes at $\mathbf{y} = \mathbf{x} - \frac{1}{\mu}\nabla f(\mathbf{x})$, with value
$f(\mathbf{x}) - \frac{1}{2\mu}\lVert \nabla f(\mathbf{x})\rVert^2$.

**Step 3 (minimize the left side over $\mathbf{y}$).** That minimum is $f^{*}$. The inequality holds
for every $\mathbf{y}$, hence between the minima:
$f^{*} \ge f(\mathbf{x}) - \frac{1}{2\mu}\lVert \nabla f(\mathbf{x})\rVert^2$.

**Step 4 (rearrange).**

$$
\boxed{\tfrac12\lVert \nabla f(\mathbf{x})\rVert^2 \ge \mu\left(f(\mathbf{x})-f^{*}\right)}
$$

The converse fails (Problem L0.4), so PL is strictly weaker than strong convexity yet powers the
same linear rate.

**Key takeaway.** The hierarchy is strong convexity $\Rightarrow$ PL $\Rightarrow$ linear rate, and
the linear-rate proof consumes only the second implication.

In [10]:
d15 = 5
B = rng.standard_normal((d15, d15))
A15 = B @ B.T + 0.7 * np.eye(d15)          # symmetric positive definite
b15 = rng.standard_normal(d15)
mu15 = np.linalg.eigvalsh(A15).min()
x_star15 = np.linalg.solve(A15, b15)
f15 = lambda x: 0.5 * x @ A15 @ x - b15 @ x
g15 = lambda x: A15 @ x - b15
fstar15 = f15(x_star15)
probes = rng.standard_normal((8, d15)) * 2
ratios = np.array([0.5 * g15(x) @ g15(x) / (f15(x) - fstar15) for x in probes])
print(f"mu = lambda_min(A) = {mu15:.6f}")
print(f"0.5|grad|^2/(f-f*) at 8 probes = {ratios}")
print(f"all >= mu ? {np.all(ratios >= mu15 - 1e-9)}   min ratio = {ratios.min():.6f}")
assert np.all(ratios >= mu15 - 1e-9)

mu = lambda_min(A) = 1.192776
0.5|grad|^2/(f-f*) at 8 probes = [10.829314 12.687969  4.295088 12.786136 11.108688  9.980584  9.017369
 14.431675]
all >= mu ? True   min ratio = 4.295088


### Problem L1.6 — The Rosenbrock Valley

**Statement.** For $f(x,y) = (1-x)^2 + 100(y-x^2)^2$ compute the Hessian at the minimizer $(1,1)$,
its condition number, and the iterations-per-digit cost of optimally tuned gradient descent near
the minimum — both in the distance and in the function value.

**Intuition.** Rosenbrock's banana valley is a curved, extremely anisotropic bowl: the canonical
stress test for first-order methods.

**Solution.**

**Step 1 (derivatives).** $f_x = -2(1-x) - 400x(y-x^2)$ and $f_y = 200(y-x^2)$, so

$$
\nabla^2 f(x,y) = \begin{bmatrix}2 + 1200x^2 - 400y & -400x\\ -400x & 200\end{bmatrix}, \qquad \nabla^2 f(1,1) = \begin{bmatrix}802 & -400\\ -400 & 200\end{bmatrix}
$$

**Step 2 (eigenvalues).** Trace $= 1002$ and determinant $= 802\cdot200 - 400^2 = 400$, so

$$
\lambda_{\pm} = \frac{1002 \pm \sqrt{1002^2 - 1600}}{2}, \qquad \lambda_{\max} \approx 1001.6006, \quad \lambda_{\min} \approx 0.39936
$$

(their product is $400$, matching the determinant).

**Step 3 (conditioning).** $\kappa \approx 1001.6006/0.39936 \approx 2508.0$, so
$\rho = \frac{\kappa-1}{\kappa+1} \approx 0.9992028$.

**Step 4 (cost, stated in both currencies).** One digit of *distance* costs
$\ln 10/(-\ln\rho) \approx 2887$ iterations. Since the value gap contracts by $\rho^2$, one digit of
$f$ costs half that, $\approx 1444$ iterations. Quoting the first number as a value rate is the
classic mistake.

$$
\boxed{\kappa(1,1) \approx 2.508\times10^3;\quad \approx 2887 \text{ iterations per digit of } \lVert \mathbf{x}_k - \mathbf{x}^{*}\rVert,\ \approx 1444 \text{ per digit of } f}
$$

**Key takeaway.** Rosenbrock is hard for gradient descent not because it is globally non-convex but
because near the solution it is a $\kappa \approx 2500$ quadratic; curvature-aware methods
neutralize $\kappa$ entirely.

In [11]:
H16 = np.array([[802.0, -400.0], [-400.0, 200.0]])
ev16 = np.linalg.eigvalsh(H16)
kap16 = ev16.max() / ev16.min()
rho16 = (kap16 - 1) / (kap16 + 1)
print(f"trace={np.trace(H16)}  det={np.linalg.det(H16):.4f}")
print(f"eigenvalues = {ev16}   product = {np.prod(ev16):.6f}")
print(f"kappa = {kap16:.4f}   rho = {rho16:.7f}")
print(f"iterations per digit, distance = {iters_per_digit(rho16):.1f}")
print(f"iterations per digit, value    = {iters_per_digit(rho16 ** 2):.1f}")

f_rb = lambda p: (1 - p[0]) ** 2 + 100 * (p[1] - p[0] ** 2) ** 2
g_rb = lambda p: np.array([-2 * (1 - p[0]) - 400 * p[0] * (p[1] - p[0] ** 2),
                           200 * (p[1] - p[0] ** 2)])
start = np.array([1.0, 1.0]) + 1e-3 * np.array([1.0, 1.0])
path = gd_path(g_rb, start, 2 / (ev16.max() + ev16.min()), 6000)
d0, d1 = np.linalg.norm(path[0] - 1), np.linalg.norm(path[-1] - 1)
print(f"measured distance contraction per step near (1,1) = {(d1 / d0) ** (1 / 6000):.7f}"
      f"   predicted {rho16:.7f}")

trace=1002.0  det=400.0000
eigenvalues = [   0.399361 1001.600639]   product = 400.000000
kappa = 2508.0096   rho = 0.9992029
iterations per digit, distance = 2887.5
iterations per digit, value    = 1443.7


measured distance contraction per step near (1,1) = 0.9992526   predicted 0.9992029


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Gradient Descent on Least Squares

**Statement.** For $f(\mathbf{w}) = \frac12\lVert X\mathbf{w}-\mathbf{y}\rVert^2$ with the
eigenvalues of $X^TX$ in $[1,100]$, give the gradient, the Hessian, $L$, $\mu$, $\kappa$, the linear
rate at $\alpha = 1/L$, and the iterations per decimal digit **of accuracy in $f$** at both
$\alpha = 1/L$ and $\alpha^{*}$.

**Intuition.** Training a linear model is gradient descent on a quadratic whose spectrum is the
spectrum of the data Gram matrix.

**Solution.**

**Step 1 (calculus).** $\nabla f(\mathbf{w}) = X^T(X\mathbf{w}-\mathbf{y})$ and
$\nabla^2 f = X^TX$, constant, so $f$ is an exact quadratic.

**Step 2 (constants).** $L = 100$, $\mu = 1$, $\kappa = 100$.

**Step 3 (guaranteed rate at $\alpha = 1/L$).** By the linear-rate theorem,

$$
f(\mathbf{w}_k) - f^{*} \le \left(1-\frac{\mu}{L}\right)^k\left(f(\mathbf{w}_0)-f^{*}\right) = (0.99)^k\left(f(\mathbf{w}_0)-f^{*}\right)
$$

so the *guarantee* costs $k \ge \frac{\ln 10}{-\ln 0.99} \approx 229.1$ iterations per digit of $f$.

**Step 4 (the actual quadratic rate, in the same currency).** On a quadratic the error contracts by
$\max_{\lambda}\lvert 1-\lambda/L\rvert = 1-\mu/L = 0.99$ per step and the value gap, being a
quadratic form in the error, contracts by the **square** $0.99^2 = 0.9801$. So the true cost at
$\alpha = 1/L$ is $\frac{\ln 10}{-\ln 0.9801} \approx 114.6$ iterations per digit — the theorem's
$229$ is a factor-two-conservative bound, not the observed behaviour.

**Step 5 (the optimal step, again in value).** $\alpha^{*} = 2/(L+\mu) = 2/101$ contracts the
**distance** by $\frac{\kappa-1}{\kappa+1} = \frac{99}{101} = 0.980198$, which alone would read
$115.1$ iterations per digit *of distance*. Squaring it for the value gap gives
$0.960788$ and

$$
k \ge \frac{\ln 10}{-\ln 0.960788} \approx 57.6 \text{ iterations per digit of } f
$$

Quoting $229$ (a value number) against $115$ (a distance number) mixes currencies; compared like
with like the optimal step is exactly twice as fast, $114.6$ against $57.6$.

$$
\boxed{\nabla^2 f = X^TX,\ \kappa = 100,\ f(\mathbf{w}_k)-f^{*} \le (0.99)^k\Delta_0;\ \text{per digit of } f: 229 \text{ guaranteed},\ 114.6 \text{ actual at } \alpha = 1/L,\ 57.6 \text{ at } \alpha^{*}}
$$

**Key takeaway.** Correlated or badly scaled features inflate $\kappa(X^TX)$ and multiply training
time one-for-one — and every rate must be quoted in one currency, value or distance, never both.

In [12]:
kap21 = 100.0
mu21, L21 = 1.0, 100.0
rate_value_1L = 1 - mu21 / L21
rate_dist_opt = (kap21 - 1) / (kap21 + 1)
print(f"alpha=1/L : GUARANTEE (1-mu/L)^k  {rate_value_1L:.6f} -> {iters_per_digit(rate_value_1L):.2f} iters/digit of f")
print(f"alpha=1/L : actual value rate      {rate_value_1L ** 2:.6f} -> {iters_per_digit(rate_value_1L ** 2):.2f} iters/digit of f")
print(f"alpha*    : distance rate          {rate_dist_opt:.6f} -> {iters_per_digit(rate_dist_opt):.2f} iters/digit of DISTANCE")
print(f"alpha*    : actual value rate      {rate_dist_opt ** 2:.6f} -> {iters_per_digit(rate_dist_opt ** 2):.2f} iters/digit of f")

n21, d21 = 300, 25
U, _ = np.linalg.qr(rng.standard_normal((d21, d21)))
G = U @ np.diag(np.linspace(mu21, L21, d21)) @ U.T          # target X^T X
X21 = sla.sqrtm(G).real
w_true = rng.standard_normal(d21)
y21 = X21 @ w_true
f21 = lambda w: 0.5 * np.sum((X21 @ w - y21) ** 2)
g21 = lambda w: X21.T @ (X21 @ w - y21)
for label, alpha, win in [("1/L", 1 / L21, (500, 2000)), ("alpha*", 2 / (L21 + mu21), (200, 800))]:
    p = gd_path(g21, np.zeros(d21), alpha, 2100)
    gaps = np.array([f21(x) for x in p])
    lo, hi = win
    assert gaps[hi - 1] > 1e-24 * gaps[0], "fit window must stay above the floating-point floor"
    fit = np.exp(np.polyfit(np.arange(lo, hi), np.log(gaps[lo:hi]), 1)[0])
    print(f"measured value-gap factor at {label:<7}= {fit:.6f} on k in {win}"
          f"   -> {iters_per_digit(fit):.2f} iterations per digit of f")

alpha=1/L : GUARANTEE (1-mu/L)^k  0.990000 -> 229.11 iters/digit of f
alpha=1/L : actual value rate      0.980100 -> 114.55 iters/digit of f
alpha*    : distance rate          0.980198 -> 115.13 iters/digit of DISTANCE
alpha*    : actual value rate      0.960788 -> 57.56 iters/digit of f
measured value-gap factor at 1/L    = 0.980100 on k in (500, 2000)   -> 114.55 iterations per digit of f


measured value-gap factor at alpha* = 0.960788 on k in (200, 800)   -> 57.56 iterations per digit of f


### Problem L2.2 — The Learning-Rate Divergence Threshold

**Statement.** A least-squares model has $\lambda_{\max}(X^TX) = 50$ and an engineer sets
$\alpha = 0.05$. Predict what happens, quantify the growth rate, and state the safe range and the
optimal fixed rate given also $\lambda_{\min}(X^TX) = 2$.

**Intuition.** Divergence in training is rarely mysterious: it is one eigenvalue crossing the
stability boundary.

**Solution.**

**Step 1 (stability check).** Along the top eigenvector the error multiplies by
$1-\alpha\lambda_{\max} = 1 - 0.05\cdot50 = -1.5$ per step, so

$$
\lvert e_k\rvert = (1.5)^k\lvert e_0\rvert, \qquad f\text{-contribution} \propto (2.25)^k
$$

After $20$ steps the amplitude has grown by $1.5^{20} \approx 3325$: the classic exploding,
oscillating loss curve.

**Step 2 (safe range).** Every mode is stable iff $0 \lt \alpha \lt 2/\lambda_{\max} = 0.04$.

**Step 3 (optimal fixed rate).** With $\mu = 2$, $L = 50$, $\kappa = 25$,

$$
\alpha^{*} = \frac{2}{L+\mu} = \frac{2}{52} \approx 0.038462, \qquad \rho = \frac{\kappa-1}{\kappa+1} = \frac{24}{26} \approx 0.923077
$$

$$
\boxed{\alpha = 0.05 \gt \tfrac{2}{L} = 0.04 \Rightarrow \text{divergence at } 1.5\text{/step}; \quad \text{safe } \alpha \lt 0.04,\ \alpha^{*} \approx 0.038462}
$$

**Key takeaway.** The largest stable learning rate is a *measurable* quantity, $2/\lambda_{\max}$;
a few power iterations turn learning-rate tuning from folklore into arithmetic.

In [13]:
L22, mu22 = 50.0, 2.0
A22 = np.diag([mu22, L22])
g22 = lambda x: A22 @ x
print(f"2/L = {2 / L22:.4f}   alpha* = {2 / (L22 + mu22):.6f}   "
      f"rho* = {(L22 / mu22 - 1) / (L22 / mu22 + 1):.6f}")
for alpha in [0.0385, 0.04, 0.05]:
    p = gd_path(g22, np.array([1.0, 1.0]), alpha, 20)
    print(f"alpha={alpha:.4f}  stiff factor {1 - alpha * L22:+.3f}  "
          f"|x_20|/|x_0| = {np.linalg.norm(p[-1]) / np.linalg.norm(p[0]):.4e}")
print(f"1.5^20 = {1.5 ** 20:.1f}")
p_bad = gd_path(g22, np.array([1.0, 1.0]), 0.05, 20)
assert abs(abs(p_bad[-1][1]) - 1.5 ** 20) < 1e-6

2/L = 0.0400   alpha* = 0.038462   rho* = 0.923077
alpha=0.0385  stiff factor -0.925  |x_20|/|x_0| = 2.0589e-01
alpha=0.0400  stiff factor -1.000  |x_20|/|x_0| = 7.1959e-01
alpha=0.0500  stiff factor -1.500  |x_20|/|x_0| = 2.3513e+03
1.5^20 = 3325.3


### Problem L2.3 — Smoothness Constant of Logistic Regression

**Statement.** For

$$
f(\mathbf{w}) = \frac1n\sum_{i=1}^n \log\left(1+e^{-y_i\mathbf{w}^T\mathbf{x}_i}\right) + \frac{\lambda}{2}\lVert \mathbf{w}\rVert^2, \qquad y_i \in \{-1,+1\}
$$

derive the Hessian, prove $L$-smoothness with $L = \frac{1}{4n}\lambda_{\max}(X^TX)+\lambda$,
identify $\mu$, and state the rate.

**Intuition.** The sigmoid can only bend so much — its derivative peaks at $1/4$ — which caps the
curvature of the whole loss.

**Solution.**

**Step 1 (gradient and Hessian).** With $\sigma(t) = 1/(1+e^{-t})$ and
$t_i = y_i\mathbf{w}^T\mathbf{x}_i$, using $\frac{d}{dt}\log(1+e^{-t}) = \sigma(t)-1$ and $y_i^2 = 1$,

$$
\nabla f(\mathbf{w}) = \frac1n\sum_i\left(\sigma(t_i)-1\right)y_i\mathbf{x}_i + \lambda\mathbf{w}, \qquad \nabla^2 f(\mathbf{w}) = \frac1n X^TSX + \lambda I
$$

with $S = \operatorname{diag}\left(\sigma(t_i)(1-\sigma(t_i))\right)$.

**Step 2 (upper bound).** $\sigma(1-\sigma) \le \frac14$, so $0 \preceq S \preceq \frac14 I$ and for
every $\mathbf{v}$,

$$
\mathbf{v}^T\nabla^2 f\,\mathbf{v} \le \frac{1}{4n}\lVert X\mathbf{v}\rVert^2 + \lambda\lVert \mathbf{v}\rVert^2 \le \left(\frac{\lambda_{\max}(X^TX)}{4n}+\lambda\right)\lVert \mathbf{v}\rVert^2
$$

**Step 3 (lower bound).** $S \succeq 0$ gives $\nabla^2 f \succeq \lambda I$, so $f$ is
$\lambda$-strongly convex: $\mu = \lambda$.

**Step 4 (rate).**

$$
\boxed{L = \frac{\lambda_{\max}(X^TX)}{4n}+\lambda,\quad \mu = \lambda,\quad f(\mathbf{w}_k)-f^{*} \le \left(1-\frac{\lambda}{L}\right)^k\Delta_0}
$$

**Key takeaway.** Regularization is not only statistical: the ridge term is what supplies
$\mu \gt 0$ and turns logistic training from a sublinear $O(1/k)$ problem into a linearly
convergent one.

In [14]:
n23, d23, lam23 = 400, 8, 0.05
X23 = rng.standard_normal((n23, d23))
y23 = np.sign(rng.standard_normal(n23))
sig = lambda t: 1.0 / (1.0 + np.exp(-t))


def f23(w):
    return np.mean(np.logaddexp(0.0, -y23 * (X23 @ w))) + 0.5 * lam23 * w @ w


def g23(w):
    t = y23 * (X23 @ w)
    return X23.T @ ((sig(t) - 1) * y23) / n23 + lam23 * w


def h23(w):
    t = y23 * (X23 @ w)
    s = sig(t) * (1 - sig(t))
    return X23.T @ (s[:, None] * X23) / n23 + lam23 * np.eye(d23)


L23 = np.linalg.eigvalsh(X23.T @ X23).max() / (4 * n23) + lam23
worst_lo, worst_hi = np.inf, -np.inf
for _ in range(200):
    w = rng.standard_normal(d23) * 3
    e = np.linalg.eigvalsh(h23(w))
    worst_lo, worst_hi = min(worst_lo, e.min()), max(worst_hi, e.max())
print(f"claimed L = lam_max(X'X)/(4n) + lambda = {L23:.6f}")
print(f"observed Hessian spectrum over 200 random w: [{worst_lo:.6f}, {worst_hi:.6f}]")
print(f"claimed mu = lambda = {lam23:.6f}")
assert worst_hi <= L23 + 1e-9 and worst_lo >= lam23 - 1e-9

grad_num = sopt.approx_fprime(np.ones(d23) * 0.3, f23, 1e-6)
print(f"analytic vs numerical gradient, rel. error = "
      f"{np.linalg.norm(grad_num - g23(np.ones(d23) * 0.3)) / np.linalg.norm(g23(np.ones(d23) * 0.3)):.3e}")
w_ref = sopt.minimize(f23, np.zeros(d23), jac=g23, method="L-BFGS-B",
                      options={"gtol": 1e-14, "ftol": 1e-18}).x
w0_23 = 3 * np.ones(d23)                      # start far out so the gap stays measurable
path23 = gd_path(g23, w0_23, 1 / L23, 12)
delta0 = f23(w0_23) - f23(w_ref)
for k in [1, 3, 6, 9, 12]:
    gap_k = f23(path23[k]) - f23(w_ref)
    bound_k = (1 - lam23 / L23) ** k * delta0
    print(f"k={k:3d}  gap = {gap_k:.4e}   Thm bound (1-mu/L)^k Delta_0 = {bound_k:.4e}   ok = {gap_k <= bound_k}")
    assert gap_k <= bound_k
print("The bound is loose because L is a worst-case cap: the observed Hessian never reaches it "
      "(see the spectrum printed above), so the realized rate beats the guarantee.")

claimed L = lam_max(X'X)/(4n) + lambda = 0.359902
observed Hessian spectrum over 200 random w: [0.050359, 0.234439]
claimed mu = lambda = 0.050000
analytic vs numerical gradient, rel. error = 1.861e-06
k=  1  gap = 2.7042e+00   Thm bound (1-mu/L)^k Delta_0 = 3.6907e+00   ok = True
k=  3  gap = 7.8189e-01   Thm bound (1-mu/L)^k Delta_0 = 2.7364e+00   ok = True
k=  6  gap = 8.5993e-03   Thm bound (1-mu/L)^k Delta_0 = 1.7470e+00   ok = True
k=  9  gap = 7.3105e-06   Thm bound (1-mu/L)^k Delta_0 = 1.1154e+00   ok = True


k= 12  gap = 7.2563e-09   Thm bound (1-mu/L)^k Delta_0 = 7.1211e-01   ok = True
The bound is loose because L is a worst-case cap: the observed Hessian never reaches it (see the spectrum printed above), so the realized rate beats the guarantee.


### Problem L2.4 — Heavy-Ball Momentum Is a Damped Oscillator (physics)

**Statement.** Show that heavy ball,
$\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\nabla f(\mathbf{x}_k) + \beta(\mathbf{x}_k-\mathbf{x}_{k-1})$,
is a finite-difference discretization of $m\ddot{\mathbf{x}} + c\dot{\mathbf{x}} = -\nabla f(\mathbf{x})$,
identify $m$ and $c$ in terms of $\alpha$, $\beta$ and the time step $h$, and interpret
$\beta \to 0$ and $\beta \to 1$.

**Intuition.** Momentum literally gives the marble on the loss surface a mass; $\beta$ tunes the
friction.

**Solution.**

**Step 1 (rewrite as differences).** Rearranging heavy ball,

$$
\left(\mathbf{x}_{k+1}-2\mathbf{x}_k+\mathbf{x}_{k-1}\right) + (1-\beta)\left(\mathbf{x}_k-\mathbf{x}_{k-1}\right) = -\alpha\nabla f(\mathbf{x}_k)
$$

(expanding the left side gives $\mathbf{x}_{k+1}-\mathbf{x}_k-\beta\mathbf{x}_k+\beta\mathbf{x}_{k-1}$,
which is the iteration).

**Step 2 (identify difference quotients).** With $\mathbf{x}_k = \mathbf{x}(kh)$,
$\ddot{\mathbf{x}} \approx h^{-2}(\mathbf{x}_{k+1}-2\mathbf{x}_k+\mathbf{x}_{k-1})$ and
$\dot{\mathbf{x}} \approx h^{-1}(\mathbf{x}_k-\mathbf{x}_{k-1})$. Substituting and dividing by
$\alpha$,

$$
\frac{h^2}{\alpha}\ddot{\mathbf{x}} + \frac{(1-\beta)h}{\alpha}\dot{\mathbf{x}} = -\nabla f(\mathbf{x})
$$

**Step 3 (read off the physics).** $m = h^2/\alpha$ and $c = (1-\beta)h/\alpha$, so the damping
ratio on a mode of curvature $\lambda$ is $c/\left(2\sqrt{m\lambda}\right) = (1-\beta)/(2\sqrt{\alpha\lambda})$.

- $\beta \to 0$: $c$ is maximal relative to $m$ — the overdamped limit, i.e. plain gradient descent
  and gradient flow.
- $\beta \to 1$: friction vanishes and the oscillator rings forever; loss curves oscillate strongly.

**Step 4 (the optimal tuning is not critical damping).** Critical damping would set
$c^2 = 4m\lambda$ for one specific $\lambda$. The optimal discrete choice
$\beta^{*} = \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2$ instead places **every** mode
$\lambda \in [\mu,L]$ on the *underdamped* boundary, where the two roots are complex with modulus
$\sqrt{\beta^{*}}$ regardless of $\lambda$ — trading a hair of ringing for a spectrum-independent
rate (Problem L3.3).

$$
\boxed{m\ddot{\mathbf{x}} + c\dot{\mathbf{x}} = -\nabla f(\mathbf{x}) \ \text{with}\ m = \frac{h^2}{\alpha},\ c = \frac{(1-\beta)h}{\alpha}}
$$

**Key takeaway.** Momentum swaps the first-order dynamics of gradient descent for genuine Newtonian
inertia: mass carries the iterate through narrow valleys, but the friction must be tuned, and the
right tuning is *under*-damping, not critical damping.

In [15]:
lam24, alpha24, beta24, h24 = 3.0, 0.05, 0.8, 1.0
m24, c24 = h24 ** 2 / alpha24, (1 - beta24) * h24 / alpha24
print(f"m = h^2/alpha = {m24:.4f},  c = (1-beta)h/alpha = {c24:.4f}")
print(f"damping ratio c/(2 sqrt(m lambda)) = {c24 / (2 * np.sqrt(m24 * lam24)):.4f}"
      f"  (=(1-beta)/(2 sqrt(alpha lambda)) = {(1 - beta24) / (2 * np.sqrt(alpha24 * lam24)):.4f})")

# heavy-ball iterates vs the ODE integrated with the SAME leapfrog stencil
x_hb = [1.0, 1.0]
for _ in range(60):
    x_hb.append(x_hb[-1] - alpha24 * lam24 * x_hb[-1] + beta24 * (x_hb[-1] - x_hb[-2]))
x_hb = np.array(x_hb[1:])
x_ode = [1.0, 1.0]
for _ in range(60):
    acc = (-lam24 * x_ode[-1] - c24 * (x_ode[-1] - x_ode[-2]) / h24) / m24
    x_ode.append(2 * x_ode[-1] - x_ode[-2] + h24 ** 2 * acc)
x_ode = np.array(x_ode[1:])
print(f"max |heavy-ball - discretized ODE| over 60 steps = {np.abs(x_hb - x_ode).max():.3e}")
assert np.abs(x_hb - x_ode).max() < 1e-12

kappa24 = 25.0
beta_star24 = ((np.sqrt(kappa24) - 1) / (np.sqrt(kappa24) + 1)) ** 2
alpha_star24 = 4 / (np.sqrt(kappa24 * 1.0) + 1.0) ** 2       # mu = 1, L = kappa
for lam in [1.0, 5.0, 25.0]:
    disc = (1 + beta_star24 - alpha_star24 * lam) ** 2 - 4 * beta_star24
    print(f"lambda={lam:5.1f}  discriminant={disc:+.3e} -> "
          f"{'complex (underdamped)' if disc <= 1e-12 else 'real'};  "
          f"|z| = {np.abs(np.roots([1, -(1 + beta_star24 - alpha_star24 * lam), beta_star24])).max():.6f}"
          f"   sqrt(beta*) = {np.sqrt(beta_star24):.6f}")

m = h^2/alpha = 20.0000,  c = (1-beta)h/alpha = 4.0000
damping ratio c/(2 sqrt(m lambda)) = 0.2582  (=(1-beta)/(2 sqrt(alpha lambda)) = 0.2582)
max |heavy-ball - discretized ODE| over 60 steps = 1.665e-16
lambda=  1.0  discriminant=+0.000e+00 -> complex (underdamped);  |z| = 0.666667   sqrt(beta*) = 0.666667
lambda=  5.0  discriminant=-9.877e-01 -> complex (underdamped);  |z| = 0.666667   sqrt(beta*) = 0.666667
lambda= 25.0  discriminant=+0.000e+00 -> complex (underdamped);  |z| = 0.666667   sqrt(beta*) = 0.666667


### Problem L2.5 — Gradient Flow as Overdamped Relaxation (physics)

**Statement.** Solve the gradient flow $\dot{\mathbf{x}}(t) = -A\mathbf{x}(t)$ for symmetric
$A \succ 0$ with $\mu = \lambda_{\min}(A)$, show that
$f(\mathbf{x}(t)) = \frac12\mathbf{x}(t)^TA\mathbf{x}(t)$ decays at least as fast as $e^{-2\mu t}$,
and explain what forward Euler adds that the continuous dynamics never had.

**Intuition.** This is a particle in a quadratic potential in the overdamped (inertia-free) limit —
relaxation of a spring network, a Debye dielectric, an RC circuit. In continuous time there is no
step size to get wrong, so the spectrum alone dictates per-mode exponential decay.

**Solution.**

**Step 1 (diagonalize).** With $A = Q\Lambda Q^T$ and $\mathbf{z} = Q^T\mathbf{x}$ the system
decouples: $\dot z_i = -\lambda_i z_i$, so $z_i(t) = e^{-\lambda_i t}z_i(0)$ and

$$
\mathbf{x}(t) = Q\,e^{-\Lambda t}Q^T\mathbf{x}_0 = e^{-At}\mathbf{x}_0
$$

Each mode is an independent relaxation with time constant $\tau_i = 1/\lambda_i$; the slowest is
$\tau_{\max} = 1/\mu$.

**Step 2 (energy decay).** In eigen-coordinates $f = \frac12\sum_i\lambda_i z_i^2$, so

$$
f(\mathbf{x}(t)) = \frac12\sum_i \lambda_i e^{-2\lambda_i t}z_i(0)^2 \le e^{-2\mu t}\cdot\frac12\sum_i\lambda_i z_i(0)^2 = e^{-2\mu t}f(\mathbf{x}_0)
$$

using $\lambda_i \ge \mu$. Equivalently $\dot f = -\lVert A\mathbf{x}\rVert^2 \le -2\mu f$:
the potential energy is a Lyapunov function and dissipation is monotone.

**Step 3 (what discretization adds).** Forward Euler replaces the exact mode factor
$e^{-\alpha\lambda_i}$, which lies in $(0,1)$ for *every* $\lambda_i \gt 0$, by $1-\alpha\lambda_i$,
which leaves $(-1,1)$ as soon as $\alpha\lambda_i \gt 2$. The step-size ceiling of gradient descent
is therefore a pure artefact of time discretization — the physics itself is unconditionally stable,
and stiffness (large $\kappa$) is what makes the artefact bite.

$$
\boxed{\mathbf{x}(t) = e^{-At}\mathbf{x}_0, \qquad f(\mathbf{x}(t)) \le e^{-2\mu t}f(\mathbf{x}_0)}
$$

**Key takeaway.** Gradient flow converges for every positive spectrum, so all step-size pathologies
of gradient descent are discretization artefacts — exactly the situation a stiff-ODE solver faces.

In [16]:
A25 = np.array([[3.0, 1.0], [1.0, 3.0]])
mu25 = np.linalg.eigvalsh(A25).min()
x025 = np.array([1.0, 0.7])
f25 = lambda x: 0.5 * x @ A25 @ x
ts = np.array([0.0, 0.25, 0.5, 1.0, 2.0, 4.0])
for t in ts:
    xt = sla.expm(-A25 * t) @ x025
    print(f"t={t:5.2f}  x(t)={xt}  f={f25(xt):.6e}  bound e^(-2 mu t) f0 = "
          f"{np.exp(-2 * mu25 * t) * f25(x025):.6e}")
    assert f25(xt) <= np.exp(-2 * mu25 * t) * f25(x025) + 1e-14

for alpha in [0.2, 0.45, 0.55]:
    euler = gd_path(lambda x: A25 @ x, x025, alpha, 40)[-1]
    exact = sla.expm(-A25 * alpha * 40) @ x025
    print(f"alpha={alpha:.2f}  stiff factor 1-alpha*L={1 - alpha * np.linalg.eigvalsh(A25).max():+.3f}"
          f"  |Euler x_40|={np.linalg.norm(euler):.4e}  |exact flow|={np.linalg.norm(exact):.4e}")

t= 0.00  x(t)=[1.  0.7]  f=2.935000e+00  bound e^(-2 mu t) f0 = 2.935000e+00
t= 0.25  x(t)=[0.403677 0.221718]  f=4.076735e-01  bound e^(-2 mu t) f0 = 1.079726e+00
t= 0.50  x(t)=[0.170217 0.059853]  f=5.902228e-02  bound e^(-2 mu t) f0 = 3.972091e-01
t= 1.00  x(t)=[ 0.035869 -0.004732]  f=1.793691e-03  bound e^(-2 mu t) f0 = 5.375640e-02
t= 2.00  x(t)=[ 0.003032 -0.002462]  f=1.542104e-05  bound e^(-2 mu t) f0 = 9.845828e-04
t= 4.00  x(t)=[ 0.00005 -0.00005]  f=5.064119e-09  bound e^(-2 mu t) f0 = 3.302907e-07
alpha=0.20  stiff factor 1-alpha*L=+0.200  |Euler x_40|=2.8357e-10  |exact flow|=2.3872e-08
alpha=0.45  stiff factor 1-alpha*L=-0.800  |Euler x_40|=1.5978e-04  |exact flow|=4.9205e-17
alpha=0.55  stiff factor 1-alpha*L=-1.200  |Euler x_40|=1.7668e+03  |exact flow|=1.6506e-20


### Problem L2.6 — The Acceleration Dividend at Scale

**Statement.** A strongly convex objective has $\kappa = 10^4$. Compare the iteration counts of
plain gradient descent and Nesterov's method to reach relative accuracy $\epsilon = 10^{-6}$, using
the rates $(1-1/\kappa)^k$ and $(1-1/\sqrt{\kappa})^k$.

**Intuition.** Acceleration replaces $\kappa$ by $\sqrt{\kappa}$ — the difference between hours and
seconds when $\kappa$ is large.

**Solution.**

**Step 1 (iteration-count formula).** $(1-q)^k \le \epsilon$ requires
$k \ge \frac{\ln(1/\epsilon)}{-\ln(1-q)} \approx \frac{\ln(1/\epsilon)}{q}$ for small $q$, and
$\ln(10^6) \approx 13.816$.

**Step 2 (plain gradient descent).** $q = 1/\kappa = 10^{-4}$, so
$k_{\mathrm{GD}} \approx 13.816\times10^4 \approx 1.38\times10^5$.

**Step 3 (Nesterov).** $q = 1/\sqrt{\kappa} = 10^{-2}$, so
$k_{\mathrm{NAG}} \approx 13.816\times10^2 \approx 1.38\times10^3$.

**Step 4 (dividend).** The ratio is $\kappa/\sqrt{\kappa} = \sqrt{\kappa} = 100$, and since both
methods cost one gradient plus $O(n)$ vector work per step, it is a genuine hundredfold saving in
wall-clock time.

$$
\boxed{k_{\mathrm{GD}} \approx 1.38\times10^5, \quad k_{\mathrm{NAG}} \approx 1.38\times10^3, \quad \text{speedup} = \sqrt{\kappa} = 100}
$$

**Key takeaway.** Acceleration obeys a square-root law: its benefit grows as $\sqrt{\kappa}$, which
is why momentum is indispensable on stiff problems and nearly irrelevant on well-conditioned ones.

In [17]:
kap26, eps26 = 1e4, 1e-6
k_gd = np.log(1 / eps26) / -np.log(1 - 1 / kap26)
k_nag = np.log(1 / eps26) / -np.log(1 - 1 / np.sqrt(kap26))
print(f"ln(1/eps) = {np.log(1 / eps26):.4f}")
print(f"k_GD  = {k_gd:.1f}  (approx {np.log(1 / eps26) * kap26:.1f})")
print(f"k_NAG = {k_nag:.1f}  (approx {np.log(1 / eps26) * np.sqrt(kap26):.1f})")
print(f"speedup = {k_gd / k_nag:.2f}   sqrt(kappa) = {np.sqrt(kap26):.1f}")
assert abs(k_gd / k_nag - np.sqrt(kap26)) / np.sqrt(kap26) < 0.02

ln(1/eps) = 13.8155
k_GD  = 138148.2  (approx 138155.1)
k_NAG = 1374.6  (approx 1381.6)
speedup = 100.50   sqrt(kappa) = 100.0


## L3 — Challenge Proofs

### Problem L3.1 — Gradient Descent as a Contraction Map

**Statement.** Let $f(\mathbf{x}) = \frac12\mathbf{x}^TA\mathbf{x}-\mathbf{b}^T\mathbf{x}$ with
$A = A^T$, $\mu I \preceq A \preceq LI$, $\mu \gt 0$. Prove $T(\mathbf{x}) = \mathbf{x}-\alpha\nabla f(\mathbf{x})$
is a contraction on $(\mathbb{R}^n, \lVert\cdot\rVert_2)$ if and only if $0 \lt \alpha \lt 2/L$,
find the optimal contraction constant, and conclude via Banach.

**Intuition.** Convergence of an iteration is a fixed-point phenomenon: shrink distances uniformly
and you must spiral into the unique fixed point.

**Solution.**

**Step 1 (the map is affine).** $\nabla f(\mathbf{x}) = A\mathbf{x}-\mathbf{b}$, so
$T(\mathbf{x}) = (I-\alpha A)\mathbf{x}+\alpha\mathbf{b}$ and
$T(\mathbf{x})-T(\mathbf{y}) = (I-\alpha A)(\mathbf{x}-\mathbf{y})$.

**Step 2 (Lipschitz constant).** $I-\alpha A$ is symmetric with eigenvalues $1-\alpha\lambda_i$, so

$$
\lVert I-\alpha A\rVert_2 = \max_i\lvert 1-\alpha\lambda_i\rvert = \max\left(\lvert 1-\alpha\mu\rvert, \lvert 1-\alpha L\rvert\right) =: \rho(\alpha)
$$

because $\lambda \mapsto \lvert 1-\alpha\lambda\rvert$ is convex on $[\mu,L]$.

**Step 3 (contraction iff $0 \lt \alpha \lt 2/L$).** Both $\lvert 1-\alpha\mu\rvert \lt 1$ and
$\lvert 1-\alpha L\rvert \lt 1$ read $0 \lt \alpha\lambda \lt 2$; since $\mu \le L$ the binding
constraint is $\alpha L \lt 2$.

**Step 4 (optimal constant).** $\rho$ is minimized where the branches cross with opposite signs,
$1-\alpha\mu = \alpha L-1$:

$$
\alpha^{*} = \frac{2}{L+\mu}, \qquad \rho(\alpha^{*}) = \frac{L-\mu}{L+\mu} = \frac{\kappa-1}{\kappa+1}
$$

**Step 5 (Banach).** $\mathbb{R}^n$ is complete and $T$ is a $\rho$-contraction with $\rho \lt 1$,
so $T$ has a unique fixed point $\bar{\mathbf{x}}$ with
$\lVert \mathbf{x}_k-\bar{\mathbf{x}}\rVert \le \rho^k\lVert \mathbf{x}_0-\bar{\mathbf{x}}\rVert$.
The fixed-point equation $\bar{\mathbf{x}} = \bar{\mathbf{x}}-\alpha(A\bar{\mathbf{x}}-\mathbf{b})$
gives $A\bar{\mathbf{x}} = \mathbf{b}$, so $\bar{\mathbf{x}} = \mathbf{x}^{*}$.

$$
\boxed{T \text{ is a contraction} \iff 0 \lt \alpha \lt \frac{2}{L}, \qquad \min_{\alpha}\rho(\alpha) = \frac{\kappa-1}{\kappa+1} \text{ at } \alpha^{*} = \frac{2}{L+\mu}}
$$

**Key takeaway.** Framing gradient descent as a fixed-point contraction unifies it with value
iteration, Picard iteration and the power method — one theorem (Banach) underwrites them all.

In [18]:
d31 = 6
B31 = rng.standard_normal((d31, d31))
A31 = B31 @ B31.T + 0.5 * np.eye(d31)
ev31 = np.linalg.eigvalsh(A31)
mu31, L31 = ev31.min(), ev31.max()
kap31 = L31 / mu31
alphas = np.linspace(1e-4, 2.4 / L31, 20001)
rho = np.array([np.linalg.norm(np.eye(d31) - a * A31, 2) for a in alphas[::200]])
a_grid = alphas[::200]
print(f"mu={mu31:.4f}  L={L31:.4f}  kappa={kap31:.4f}")
print(f"argmin ||I - alpha A||_2 = {a_grid[rho.argmin()]:.6f}   theory 2/(L+mu) = {2 / (L31 + mu31):.6f}")
print(f"min ||I - alpha A||_2    = {rho.min():.6f}   theory (kappa-1)/(kappa+1) = {(kap31 - 1) / (kap31 + 1):.6f}")
print(f"||I - alpha A||_2 at alpha slightly above 2/L: "
      f"{np.linalg.norm(np.eye(d31) - 1.01 * (2 / L31) * A31, 2):.6f} (> 1, not a contraction)")
b31 = rng.standard_normal(d31)
xs31 = gd_path(lambda x: A31 @ x - b31, np.zeros(d31), 2 / (L31 + mu31), 400)
x_fix = np.linalg.solve(A31, b31)
print(f"|x_400 - A^{{-1}}b| = {np.linalg.norm(xs31[-1] - x_fix):.3e}")
assert np.linalg.norm(np.eye(d31) - (2 / (L31 + mu31)) * A31, 2) < 1

mu=0.6009  L=12.3533  kappa=20.5589
argmin ||I - alpha A||_2 = 0.153502   theory 2/(L+mu) = 0.154390
min ||I - alpha A||_2    = 0.907764   theory (kappa-1)/(kappa+1) = 0.907231
||I - alpha A||_2 at alpha slightly above 2/L: 1.020000 (> 1, not a contraction)
|x_400 - A^{-1}b| = 2.680e-16


### Problem L3.2 — Krylov Subspaces and the First-Order Lower Bound

**Statement.** Consider methods on $f(\mathbf{x}) = \frac12\mathbf{x}^TA\mathbf{x}-\mathbf{b}^T\mathbf{x}$
that start at $\mathbf{x}_0 = \mathbf{0}$ and choose each iterate from
$\mathcal{K}_k = \operatorname{span}\{\mathbf{b}, A\mathbf{b}, \dots, A^{k-1}\mathbf{b}\}$. Show the
error has the form $-p(A)\mathbf{x}^{*}$ with $\deg p \le k$ and $p(0) = 1$, and explain why the
Chebyshev minimax value implies no such method beats the accelerated rate.

**Intuition.** Everything a first-order method can build after $k$ steps is a degree-$k$ polynomial
in $A$ applied to the data, so optimal algorithms are optimal polynomials in disguise.

**Solution.**

**Step 1 (iterates are polynomials in $A$).** By induction: $\mathbf{x}_0 = \mathbf{0}$, and the
gradient at a point of $\mathcal{K}_k$ is $A\mathbf{x}-\mathbf{b} \in \mathcal{K}_{k+1}$. Any
combination of past gradients therefore gives $\mathbf{x}_k = q_{k-1}(A)\mathbf{b}$ with
$\deg q_{k-1} \le k-1$.

**Step 2 (error polynomial).** Using $\mathbf{b} = A\mathbf{x}^{*}$,

$$
\mathbf{x}_k - \mathbf{x}^{*} = q_{k-1}(A)A\mathbf{x}^{*}-\mathbf{x}^{*} = -\left(I - A q_{k-1}(A)\right)\mathbf{x}^{*} = -p(A)\mathbf{x}^{*}
$$

with $p(\lambda) = 1-\lambda q_{k-1}(\lambda)$, $\deg p \le k$, $p(0) = 1$. Diagonalizing $A$,

$$
\frac{\lVert \mathbf{x}_k-\mathbf{x}^{*}\rVert}{\lVert \mathbf{x}_0-\mathbf{x}^{*}\rVert} \le \max_{\lambda \in [\mu,L]}\lvert p(\lambda)\rvert
$$

and an adversary who places the spectrum and the mass of $\mathbf{x}^{*}$ at the maximizer of
$\lvert p\rvert$ makes this an equality — which needs $A$ to have enough distinct eigenvalues, i.e.
$k \le n-1$.

**Step 3 (the Chebyshev extremal polynomial).** The normalized minimax problem
$\min_{\deg p \le k,\, p(0)=1}\max_{[\mu,L]}\lvert p\rvert$ is solved by the shifted, scaled
Chebyshev polynomial

$$
p_k^{*}(\lambda) = T_k\!\left(\frac{L+\mu-2\lambda}{L-\mu}\right)\Big/ T_k\!\left(\frac{L+\mu}{L-\mu}\right)
$$

with minimax value $\frac{2c^k}{1+c^{2k}}$, $c = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}$ — a
consequence of equioscillation: a competitor with a strictly smaller sup-norm would have to cross
$p_k^{*}$ at $k+1$ points while agreeing with it at $0$, impossible for degree $k$.

**Step 4 (consequence).** Since $\frac{2c^k}{1+c^{2k}} \ge c^k$, no Krylov-restricted method
converges faster than $c^k$ uniformly over spectra in $[\mu,L]$; Chebyshev iteration and tuned
heavy ball attain it, so both are optimal.

$$
\boxed{\inf_{\text{Krylov-restricted},\ k \le n-1}\ \sup_{\operatorname{spec}(A) \subset [\mu,L]} \frac{\lVert \mathbf{x}_k-\mathbf{x}^{*}\rVert}{\lVert \mathbf{x}_0-\mathbf{x}^{*}\rVert} = \frac{2c^k}{1+c^{2k}}, \qquad c = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}}
$$

**Hypotheses that must not be dropped:** the iterates are confined to the Krylov subspace (so the
argument says nothing about methods using second-order information), $\mathbf{x}_0 = \mathbf{0}$
(otherwise translate), and $k \le n-1$ (in $n$ steps the minimal polynomial of $A$ makes the error
exactly zero, so the identity becomes a strict inequality).

**Key takeaway.** Acceleration is a theorem about polynomials: gradient descent uses
$(1-\alpha\lambda)^k$, momentum builds the Chebyshev polynomial, and Chebyshev is provably the best
degree-$k$ polynomial there is.

In [19]:
def cheb_minimax(mu, L, k, grid=400001):
    """max over [mu, L] of the normalized Chebyshev polynomial with p(0) = 1."""
    lam = np.linspace(mu, L, grid)
    z = (L + mu - 2 * lam) / (L - mu)                 # in [-1, 1]
    num = np.abs(np.cos(k * np.arccos(np.clip(z, -1, 1))))
    z0 = (L + mu) / (L - mu)                          # > 1
    den = np.cosh(k * np.arccosh(z0))
    return num.max() / den


for kap in [4.0, 100.0]:
    c = (np.sqrt(kap) - 1) / (np.sqrt(kap) + 1)
    for k in [1, 2, 3, 5, 8]:
        val = cheb_minimax(1.0, kap, k)
        formula = 2 * c ** k / (1 + c ** (2 * k))
        print(f"kappa={kap:6.1f} k={k}  minimax={val:.8f}  2c^k/(1+c^2k)={formula:.8f}"
              f"  GD (kappa-1)/(kappa+1))^k={((kap - 1) / (kap + 1)) ** k:.8f}")
        assert abs(val - formula) < 1e-6

# Krylov structure: any GD/momentum iterate really is p(A) x* with p(0) = 1
n32 = 7
A32 = np.diag(np.linspace(1.0, 10.0, n32))
x_star32 = np.ones(n32)
b32 = A32 @ x_star32
xs32 = gd_path(lambda x: A32 @ x - b32, np.zeros(n32), 2 / (10.0 + 1.0), 4)
Kry = np.column_stack([np.linalg.matrix_power(A32, j) @ b32 for j in range(4)])
for k in range(1, 5):
    coef, *_ = np.linalg.lstsq(Kry[:, :k], xs32[k], rcond=None)
    print(f"k={k}: x_k lies in K_k ? residual = {np.linalg.norm(Kry[:, :k] @ coef - xs32[k]):.3e}")

kappa=   4.0 k=1  minimax=0.60000000  2c^k/(1+c^2k)=0.60000000  GD (kappa-1)/(kappa+1))^k=0.60000000
kappa=   4.0 k=2  minimax=0.21951220  2c^k/(1+c^2k)=0.21951220  GD (kappa-1)/(kappa+1))^k=0.36000000


kappa=   4.0 k=3  minimax=0.07397260  2c^k/(1+c^2k)=0.07397260  GD (kappa-1)/(kappa+1))^k=0.21600000
kappa=   4.0 k=5  minimax=0.00823031  2c^k/(1+c^2k)=0.00823031  GD (kappa-1)/(kappa+1))^k=0.07776000
kappa=   4.0 k=8  minimax=0.00030483  2c^k/(1+c^2k)=0.00030483  GD (kappa-1)/(kappa+1))^k=0.01679616
kappa= 100.0 k=1  minimax=0.98019802  2c^k/(1+c^2k)=0.98019802  GD (kappa-1)/(kappa+1))^k=0.98019802


kappa= 100.0 k=2  minimax=0.92453542  2c^k/(1+c^2k)=0.92453542  GD (kappa-1)/(kappa+1))^k=0.96078816


kappa= 100.0 k=3  minimax=0.84263843  2c^k/(1+c^2k)=0.84263843  GD (kappa-1)/(kappa+1))^k=0.94176265


kappa= 100.0 k=5  minimax=0.64639974  2c^k/(1+c^2k)=0.64639974  GD (kappa-1)/(kappa+1))^k=0.90483440
kappa= 100.0 k=8  minimax=0.38606344  2c^k/(1+c^2k)=0.38606344  GD (kappa-1)/(kappa+1))^k=0.85213924
k=1: x_k lies in K_k ? residual = 0.000e+00
k=2: x_k lies in K_k ? residual = 1.583e-15
k=3: x_k lies in K_k ? residual = 4.341e-14
k=4: x_k lies in K_k ? residual = 1.081e-12


### Problem L3.3 — The Optimal Heavy-Ball Rate

**Statement.** On $f(\mathbf{x}) = \frac12\mathbf{x}^TA\mathbf{x}$ with spectrum in $[\mu,L]$,
analyze heavy ball mode by mode, find $(\alpha^{*},\beta^{*})$ minimizing the worst-case spectral
radius, and prove the optimal contraction is
$\sqrt{\beta^{*}} = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}$.

**Intuition.** Tune the friction so that every eigenmode is (just barely) underdamped — then all
modes decay at the same universal rate.

**Solution.**

**Step 1 (scalar recursions).** In the eigenbasis, the error along eigenvalue $\lambda$ obeys
$e_{k+1} = (1+\beta-\alpha\lambda)e_k - \beta e_{k-1}$, i.e. the companion matrix
$M(\lambda) = \begin{bmatrix}1+\beta-\alpha\lambda & -\beta\\ 1 & 0\end{bmatrix}$ with
characteristic equation

$$
z^2 - (1+\beta-\alpha\lambda)z + \beta = 0
$$

**Step 2 (complex roots have modulus $\sqrt{\beta}$).** The product of the roots is $\beta$. If the
discriminant is non-positive, $(1+\beta-\alpha\lambda)^2 \le 4\beta$, the roots are complex
conjugates and $\lvert z\rvert = \sqrt{z\bar z} = \sqrt{\beta}$ — independent of $\lambda$.

**Step 3 (cover the whole spectrum).** The complex-root condition is equivalent to
$\left(1-\sqrt{\alpha\lambda}\right)^2 \le \beta \le \left(1+\sqrt{\alpha\lambda}\right)^2$; the
upper bound is automatic for $\beta \lt 1$, so we need

$$
\beta \ge \max\left(\left(1-\sqrt{\alpha\mu}\right)^2, \left(1-\sqrt{\alpha L}\right)^2\right)
$$

Making the required $\beta$ smallest balances the two terms: $1-\sqrt{\alpha\mu} = \sqrt{\alpha L}-1$,
i.e. $\sqrt{\alpha}(\sqrt{L}+\sqrt{\mu}) = 2$, so $\alpha^{*} = \frac{4}{(\sqrt{L}+\sqrt{\mu})^2}$.

**Step 4 (optimal momentum and rate).**

$$
\sqrt{\beta^{*}} = 1-\sqrt{\alpha^{*}\mu} = \frac{\sqrt{L}-\sqrt{\mu}}{\sqrt{L}+\sqrt{\mu}} = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}
$$

**Step 5 (conclusion).** Every eigenmode sits exactly on the underdamped boundary, so the asymptotic
per-step contraction is the same for all of them:

$$
\boxed{\rho_{\mathrm{HB}} = \sqrt{\beta^{*}} = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1} \quad\text{versus}\quad \rho_{\mathrm{GD}} = \frac{\kappa-1}{\kappa+1}}
$$

matching the Chebyshev lower bound of Problem L3.2: on quadratics heavy ball is an optimal
first-order method. (Off quadratics it is not even guaranteed to converge — see Lessard, Recht and
Packard, 2016.)

**Key takeaway.** The magic of momentum is the identity $\lvert z\rvert = \sqrt{\beta}$ for complex
roots: once the friction places all modes in the ringing regime, the decay rate decouples from the
eigenvalue and the whole spectrum flattens to one rate.

In [20]:
def hb_radius(alpha, beta, lams):
    """max_lambda |z| for z^2 - (1+beta-alpha*lambda) z + beta = 0, vectorized over lambda."""
    c = 1 + beta - alpha * np.asarray(lams)
    disc = c ** 2 - 4 * beta
    real = (np.abs(c) + np.sqrt(np.maximum(disc, 0.0))) / 2
    return np.where(disc <= 0, np.sqrt(beta), real).max()


for kap in [4.0, 25.0, 100.0, 1000.0]:
    mu33, L33 = 1.0, kap
    alpha33 = 4 / (np.sqrt(L33) + np.sqrt(mu33)) ** 2
    beta33 = ((np.sqrt(kap) - 1) / (np.sqrt(kap) + 1)) ** 2
    lams = np.linspace(mu33, L33, 2001)
    c = 1 + beta33 - alpha33 * lams
    print(f"kappa={kap:7.1f}  alpha*={alpha33:.6f}  beta*={beta33:.6f}  "
          f"max discriminant over the spectrum = {(c ** 2 - 4 * beta33).max():+.2e} (<= 0: all modes "
          f"underdamped)  worst |z| = {hb_radius(alpha33, beta33, lams):.6f}  "
          f"sqrt(beta*) = {np.sqrt(beta33):.6f}  GD rate {(kap - 1) / (kap + 1):.6f}")
    assert abs(hb_radius(alpha33, beta33, lams) - np.sqrt(beta33)) < 1e-6

# a grid search confirms (alpha*, beta*) really is the minimizer of the worst-case radius
kap = 100.0
lams = np.linspace(1.0, kap, 400)
a_grid = np.linspace(0.005, 0.06, 221)
b_grid = np.linspace(0.0, 0.98, 393)
radii = np.array([[hb_radius(a, bt, lams) for bt in b_grid] for a in a_grid])
i, j = np.unravel_index(radii.argmin(), radii.shape)
print(f"grid search best radius {radii.min():.6f} at (alpha, beta) = ({a_grid[i]:.6f}, {b_grid[j]:.6f})")
print(f"theory                  {(np.sqrt(kap) - 1) / (np.sqrt(kap) + 1):.6f} at (alpha*, beta*) = "
      f"({4 / (np.sqrt(kap) + 1) ** 2:.6f}, {((np.sqrt(kap) - 1) / (np.sqrt(kap) + 1)) ** 2:.6f})")
assert radii.min() >= (np.sqrt(kap) - 1) / (np.sqrt(kap) + 1) - 1e-6

kappa=    4.0  alpha*=0.444444  beta*=0.111111  max discriminant over the spectrum = +1.11e-16 (<= 0: all modes underdamped)  worst |z| = 0.333333  sqrt(beta*) = 0.333333  GD rate 0.600000
kappa=   25.0  alpha*=0.111111  beta*=0.444444  max discriminant over the spectrum = +0.00e+00 (<= 0: all modes underdamped)  worst |z| = 0.666667  sqrt(beta*) = 0.666667  GD rate 0.923077
kappa=  100.0  alpha*=0.033058  beta*=0.669421  max discriminant over the spectrum = +4.44e-16 (<= 0: all modes underdamped)  worst |z| = 0.818182  sqrt(beta*) = 0.818182  GD rate 0.980198
kappa= 1000.0  alpha*=0.003759  beta*=0.881145  max discriminant over the spectrum = +8.88e-16 (<= 0: all modes underdamped)  worst |z| = 0.938693  sqrt(beta*) = 0.938693  GD rate 0.998002


grid search best radius 0.818535 at (alpha, beta) = (0.033000, 0.670000)
theory                  0.818182 at (alpha*, beta*) = (0.033058, 0.669421)


### Problem L3.4 — PL Without Strong Convexity

**Statement.** Let $f(\mathbf{w}) = \frac12\lVert X\mathbf{w}-\mathbf{y}\rVert^2$ with
$X \in \mathbb{R}^{m\times n}$, $n \gt m$, $\operatorname{rank}(X) = m$ and
$\mathbf{y} \in \operatorname{range}(X)$. Show $f$ is not strongly convex, satisfies PL with
$\mu = \sigma_{\min}^2(X)$, and converges linearly to a global minimizer.

**Intuition.** Over-parameterized models have flat directions (no strong convexity), but the
gradient cannot vanish while the *residual* is non-zero — which is all PL asks.

**Solution.**

**Step 1 (no strong convexity).** $\nabla^2 f = X^TX \in \mathbb{R}^{n\times n}$ has rank
$m \lt n$, so $\lambda_{\min}(X^TX) = 0$ and no $\mu \gt 0$ satisfies $\nabla^2 f \succeq \mu I$.
Indeed $f$ is constant along $\ker X$, and the minimizer set is the affine subspace
$\{\mathbf{w} : X\mathbf{w} = \mathbf{y}\}$, non-empty because
$\mathbf{y} \in \operatorname{range}(X)$, with $f^{*} = 0$.

**Step 2 (gradient in terms of the residual).** With $\mathbf{r} = X\mathbf{w}-\mathbf{y}$,
$f(\mathbf{w})-f^{*} = \frac12\lVert \mathbf{r}\rVert^2$ and $\nabla f(\mathbf{w}) = X^T\mathbf{r}$.

**Step 3 (lower-bound the gradient).** $\operatorname{rank}(X) = m$ makes $XX^T$ positive definite
with $\lambda_{\min}(XX^T) = \sigma_{\min}^2(X) \gt 0$, so

$$
\lVert \nabla f(\mathbf{w})\rVert^2 = \mathbf{r}^TXX^T\mathbf{r} \ge \sigma_{\min}^2(X)\lVert \mathbf{r}\rVert^2 = 2\sigma_{\min}^2(X)\left(f(\mathbf{w})-f^{*}\right)
$$

**Step 4 (PL and the rate).** Halving gives PL with $\mu = \sigma_{\min}^2(X)$; since
$L = \sigma_{\max}^2(X)$,

$$
f(\mathbf{w}_k) \le \left(1-\frac{\sigma_{\min}^2(X)}{\sigma_{\max}^2(X)}\right)^k f(\mathbf{w}_0)
$$

**Step 5 (implicit bias).** Every gradient $X^T\mathbf{r}$ lies in $\operatorname{range}(X^T) = (\ker X)^{\perp}$,
so the iterates never leave $\mathbf{w}_0 + \operatorname{range}(X^T)$ and converge to the minimizer
nearest $\mathbf{w}_0$ — the minimum-norm interpolant when $\mathbf{w}_0 = \mathbf{0}$.

$$
\boxed{f \text{ is PL with } \mu = \sigma_{\min}^2(X) \text{ despite } \lambda_{\min}(\nabla^2 f) = 0 \Rightarrow \text{linear convergence to an interpolating solution}}
$$

**Key takeaway.** Over-parameterization destroys strong convexity but preserves gradient dominance
along the directions that matter — the modern template for proving fast training of wide networks,
whose tangent-kernel Gram matrix plays the role of $XX^T$.

In [21]:
m34, n34 = 6, 15
X34 = rng.standard_normal((m34, n34))
w_true34 = rng.standard_normal(n34)
y34 = X34 @ w_true34                                   # y in range(X): f* = 0
sv = np.linalg.svd(X34, compute_uv=False)
mu34, L34 = sv.min() ** 2, sv.max() ** 2
f34 = lambda w: 0.5 * np.sum((X34 @ w - y34) ** 2)
g34 = lambda w: X34.T @ (X34 @ w - y34)

print(f"Hessian rank = {np.linalg.matrix_rank(X34.T @ X34)} of {n34}  -> "
      f"lambda_min(X'X) = {np.linalg.eigvalsh(X34.T @ X34).min():.2e}: not strongly convex")
print(f"sigma_min^2 = {mu34:.6f}   sigma_max^2 = {L34:.6f}   kappa = {L34 / mu34:.4f}")
probes = rng.standard_normal((6, n34)) * 2
pl = np.array([0.5 * g34(w) @ g34(w) / f34(w) for w in probes])
print(f"0.5|grad|^2/(f-f*) at 6 probes = {pl}  all >= sigma_min^2 ? {np.all(pl >= mu34 - 1e-9)}")
assert np.all(pl >= mu34 - 1e-9)

path34 = gd_path(g34, np.zeros(n34), 1 / L34, 400)
gaps34 = np.array([f34(w) for w in path34])
fit34 = np.exp(np.polyfit(np.arange(50, 300), np.log(gaps34[50:300]), 1)[0])
print(f"observed per-step factor {fit34:.6f}   PL bound (1 - mu/L) = {1 - mu34 / L34:.6f}")
w_min_norm = np.linalg.pinv(X34) @ y34
print(f"|w_400 - minimum-norm solution| = {np.linalg.norm(path34[-1] - w_min_norm):.3e}"
      f"   residual |Xw-y| = {np.linalg.norm(X34 @ path34[-1] - y34):.3e}")
assert fit34 <= 1 - mu34 / L34 + 1e-9

Hessian rank = 6 of 15  -> lambda_min(X'X) = -2.30e-15: not strongly convex
sigma_min^2 = 4.050634   sigma_max^2 = 22.587786   kappa = 5.5764
0.5|grad|^2/(f-f*) at 6 probes = [13.849796 17.884114 11.316834 17.408447 14.492629 14.096456]  all >= sigma_min^2 ? True
observed per-step factor 0.800893   PL bound (1 - mu/L) = 0.820672
|w_400 - minimum-norm solution| = 1.347e-15   residual |Xw-y| = 2.220e-16
